In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft
from scipy.signal import fftconvolve, welch
from scipy.optimize import curve_fit
from scipy.stats import norm
import functions as f
import matplotlibcolors as matplotlibcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
plt.style.use('matplotlibrc')

%matplotlib widget

In [2]:
kid = 5
pread = 113
file_type = 'vis'
pw = 1500
pw_offset = 100
filter_type = 'exp'
lifetime = 200
tqp = [200, 600]
iterate = 0

dir_on = r"D:\Data\LT218Chip1_BF_20221103_MIR3_8\12KIDs mono on 3800 long\TD_Power"
dir_off = r"D:\Data\LT218Chip1_BF_20221103_MIR3_8\12KIDs mono off\TD_Power"
chuncksize = 10
nr_chuncks = 1
mph = np.array([5,40])
mpp = mph[0]


In [ ]:
pulse_files, info_files = f.get_files(dir_on, kid, pread, type=file_type)
nr_files = len(pulse_files)
f0, Q, Qc, Qi, S21_min, dt, T = f.get_info(info_files[0])
sff = 1 / dt / 1e6
sw = int(lifetime * sff)
exp_filter = f.get_window(filter_type, sw)
fig, ax = plt.subplots(figsize=(3,3),constrained_layout=True)
ax.plot(exp_filter)

In [4]:
amp, phase, _ = f.get_data(pulse_files[:chuncksize])
signal = f.coord_transformation(phase, amp, coord='circle', response='phase')

In [5]:
std = f.get_sigma(signal, exp_filter)
ph = mph*std
pp = mpp*std
locs, props = f.find_pks(signal, 2*ph[0], pp, exp_filter)
too_high_chunck = props['peak_heights'] >= ph[1]
args = np.argwhere(~too_high_chunck).flatten()
pulses, single_idx = f.get_single_pulses(signal, locs, pw, pw_offset, args)
pulse_template = np.mean(pulses, axis=0)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(3,3))
t = np.arange(0, pw+pw_offset, 1)
ax.plot(t, pulses.T, alpha=.2, lw=.1, c='y', label='_nolegend_')
ax.plot(t, pulses[0], lw=.1, c='y', label='%d pulses' % len(pulses))
ax.plot(t, pulse_template, label='mean pulse')
ax.set_xlim([t[0], t[-1]])
# ax.set_ylim([-.5, 2])
ax.set_xlabel('Time [$\mu$s]')
ax.set_ylabel('$\\theta$ [rad]')
_ = ax.legend()

fit_tqp = [150,600]    
tqp, _, popt = f.fit_decaytime(pulse_template, fit_tqp)
tqp /= sff
tqp_x = np.linspace(fit_tqp[0]/sff, fit_tqp[1]/sff, int((fit_tqp[1] - fit_tqp[0])))
fit_x = np.arange(len(tqp_x))
tqp_y = f.exp_decay(fit_x, *popt)
ax.plot(tqp_x, tqp_y, ls='--', lw=2, label='$\\tau_{qp}$=%d $\mu$s' % tqp, zorder=3)
ax.legend(bbox_to_anchor=(0., 1, 1., .102), loc='lower left',
        ncols=2, mode="expand", borderaxespad=0.)
# BEGIN: Add inset with semilogy axes

ax_inset = inset_axes(ax, width="50%", height="50%", loc='upper right')
ax_inset.plot(t, pulse_template, label='mean pulse')
ax_inset.set_xlim([t[0], t[-1]])
# ax_inset.set_ylim([1e-3, 2])
ax_inset.set_yscale('log')
# ax_inset.set_xlabel('Time [$\mu$s]')
# ax_inset.set_ylabel('$\\theta$ [rad]')
ax_inset.plot(tqp_x, tqp_y, ls='--', lw=2, label='$\\tau_{qp}$=%d $\mu$s' % tqp, zorder=3)
# END: Add inset with semilogy axes

In [7]:
noise_files, _ = f.get_files(dir_on, kid, pread, type=file_type)
amp, phase, _ = f.get_data(noise_files[:chuncksize])
noise = f.coord_transformation(phase, amp, coord='circle', response='phase')

In [8]:
std = f.get_sigma(signal, exp_filter)
ph = mph*std
pp = mpp*std
noise_locs, _ = f.find_pks(signal, ph[0], pp, exp_filter)
noises = f.get_single_noises(signal, noise_locs, pw+pw_offset)
freqs, noise_psd = f.get_avg_psd(noises, pw+pw_offset, sff, exclude_dc=False, onesided=False)
# noises_psd = 1/(1e6*N)*np.abs(fft(noises))**2
# noises_psd[:, 1:L-1] *= 2
# avg_psd_noises = np.mean(noises_psd, axis=0)
# noise_psd = np.mean(noises_psd, axis=0)

In [ ]:
opt_filter = f.opt_filter(pulse_template, noise_psd, exclude_dc=False).real
opt_filter /= np.sum(opt_filter)
fig, ax = plt.subplots(constrained_layout=True, figsize=(3,3))
ax.plot(opt_filter)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(6,4))
t = np.arange(0, pw+pw_offset, 1)
ax.plot(t, noises.T, alpha=.1, lw=.1, c='y')
# ax.plot(t, pulse_template)
ax.set_xlim([t[0], t[-1]])
ax.set_ylim([-.5, 2])
ax.set_xlabel('Time [$\mu$s]')
ax.set_ylabel('$\\theta$ [rad]')

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(6,4))
t = np.arange(0, pw+pw_offset, 1)
ax.hist(noises.flatten(), bins='auto')
ax.set_yscale('log')	
H_opt, _, _, _ = f.optimal_filter(noises, pulse_template, sff*1e6, 1, noise_psd)
ax.hist(H_opt, bins='auto')
window_length = pw + pw_offset
H_opt = np.array([f.optimal_filter(noises.flatten()[i:i + window_length].reshape((1, window_length)), pulse_template, sff*1e6, 1, noise_psd)[0] for i in range(len(noises.flatten()[:10000]) - window_length + 1)])
ax.hist(H_opt, bins='auto')
# ax.plot(t, pulse_template)
# ax.set_xlim([t[0], t[-1]])
# ax.set_ylim([-.5, 2])
# ax.set_xlabel('Time [$\mu$s]')
# ax.set_ylabel('$\\theta$ [rad]')

In [22]:
def get_pulses(pulse_files, ph, pp, pw, pw_offset, chuncksize, nr_chuncks, window):
    if nr_chuncks:
        nr_req_files = np.amin((chuncksize * nr_chuncks, nr_files))
    else:
        nr_req_files = nr_files
    analysed_files = 0
    pulses = []
    single_idx = []
    too_high_idx = []
    single_locs = []
    while analysed_files < nr_req_files:
        if analysed_files + chuncksize > nr_files:
            chuncksize = nr_files - analysed_files
        amp, phase, _ = f.get_data(pulse_files[analysed_files:analysed_files+chuncksize])
        signal = f.coord_transformation(phase, amp, coord='circle', response='phase')
        len_file = int(len(signal) / chuncksize)
        locs, props = f.find_pks(signal, ph, pp, window)
        # too_high_chunck = props['peak_heights'] >= ph[1]
        args = np.argwhere(~too_high_chunck).flatten()
        args = np.arange(len(locs), dtype=int)
        pulses_chunck, single_idx_chunck = f.get_single_pulses(signal, locs, pw, pw_offset, args)
        if len(pulses_chunck):
            pulses.append(pulses_chunck)
            single_idx.append(single_idx_chunck)
            too_high_idx.append(too_high_chunck)
            single_locs.append(locs[single_idx_chunck] + len_file*analysed_files)
        analysed_files += chuncksize
        print('Analysed %d out of %d files' % (analysed_files, nr_req_files), end='\r')
    single_idx = np.hstack(single_idx)
    pulses = np.vstack(pulses)
    too_high_idx = np.hstack(too_high_idx)
    single_locs = np.hstack(single_locs)
    return pulses, single_idx, too_high_idx, single_locs


In [ ]:
std = f.get_sigma(noise, exp_filter)
nr_stds = [16]
H_opts = []
nr_chuncks = [1, None]
for i, nr_std in enumerate(nr_stds):
    ph = nr_std * std
    pp = nr_std * std
    pulses, _, _, _ = get_pulses(pulse_files, ph, pp, pw, pw_offset, 50, nr_chuncks[i], exp_filter)
    H_opt, R_sn, _, _ = f.optimal_filter(pulses, pulse_template, sff*1e6, 1, noise_psd)
    H_opts.append(H_opt)
    print(R_sn)

In [ ]:
norm_pulse = pulse_template / np.amax(pulse_template)
sf = 1e6
N = len(norm_pulse)
L = int(N/2+1)
df = sf/N
pulse_psd = 1/(1e6*N)*np.abs(fft(pulse_template)[:L])**2
pulse_psd[1:L-1] *= 2
norm_psd = 1/(1e6*N)*np.abs(fft(norm_pulse)[:L])**2
norm_psd[1:L-1] *= 2
pulses_psd = 1/(1e6*N)*np.abs(fft(pulses)[:, :L])**2
pulses_psd[:, 1:L-1] *= 2
noises_psd = 1/(1e6*N)*np.abs(fft(noises)[:, :L])**2
noises_psd[:, 1:L-1] *= 2
avg_psd_noises = np.mean(noises_psd, axis=0)
avg_psd_pulses = np.mean(pulses_psd, axis=0)
freqs = np.arange(0, sf/2+df, df)

Rsn = np.mean(H_opt)/2.355 * np.sqrt(np.sum(2*avg_psd_noises/avg_psd_pulses))
print(Rsn)
fig, ax = plt.subplots()
ax.semilogx(freqs, 10*np.log10(pulse_psd), label='Noise PSD')
ax.semilogx(freqs, 10*np.log10(avg_psd_pulses), label='Noise PSD')
ax.semilogx(freqs, 10*np.log10(avg_psd_noises), label='Noise PSD')

In [ ]:
fig, axes = plt.subplot_mosaic('ab', constrained_layout=True, figsize=(18.5/2.54,6/2.54))
ax = axes['a']
binsize = 0.03
nr_stds_pulse_filering = 4
bins=np.arange(-.5, 2, binsize)
noise = fftconvolve(noises.flatten(), opt_filter, mode='valid')
ax.hist(noise, bins=bins, facecolor='gray', label='Noise', zorder=0)
colors = ['b', 'y', 'o'] 
alphas = [.75, .75, 1]
for i, H_opt in enumerate(H_opts):
    _ = ax.hist(H_opt, bins=bins, facecolor=colors[i],alpha=.75, label='%d$\sigma$' % (nr_stds[i]), zorder=i)

ax.set_ylim([0, 400])
ax.set_xlim([-.25,2])
ax.set_ylabel('Counts')
ax.set_xlabel('Pulse heights [rad]')


H_opt = H_opts[-1]
c = colors[len(H_opts)-1]

ax = axes['b']
t = np.arange(0, pw+pw_offset, 1)
mean_pulse = np.mean(pulses, axis=0)    
std_pulse = np.std(pulses, axis=0)
ax.plot(t, mean_pulse, c=c, label='Mean pulse main')
ax.fill_between(t, mean_pulse - std_pulse, mean_pulse + std_pulse, color=c, alpha=.2, label='$\pm1$ std')

def tau_qp(x, a, tqp):
    return a*np.exp(-x/tqp)


fit_tqp = [150,1000]    
x = np.arange(fit_tqp[0], fit_tqp[1])
popt, pcov = curve_fit(tau_qp, t[fit_tqp[0]:fit_tqp[1]], mean_pulse[fit_tqp[0]:fit_tqp[1]], p0=[1, 200])
print(popt)
tqp = popt[1]
perr = np.sqrt(np.diag(pcov))
print(tqp, perr[1])
ax.plot(x, tau_qp(x, *popt), c='k', ls='--', lw=1.5, label='fit $\\tau_{qp}$', zorder=3)

ax_inset = inset_axes(ax, width="60%", height="60%", loc='upper right')
ax_inset.semilogy(t, mean_pulse, c=c, label='mean pulse')
# ax_inset.semilogy(tqp_x, tqp_y, c='k', ls='--', lw=1)
ax_inset.semilogy(x, tau_qp(x, *popt), c='k', ls='--', lw=1, label='fit $\\tau_{qp}$', zorder=3)
ax_inset.set_xlim([t[0], 1500])
ax_inset.set_ylim([1e-3, 2])
ax_inset.set_xticklabels(np.arange(0,1500, 250))
ax.set_xticks(np.arange(0,1750, 250))
ax.set_xticklabels(np.arange(0,1750, 250))
ax.set_xlim([t[0], 1500])
ax.set_ylim([-.25, 2])
ax.set_ylabel('Pulse heights [rad]')
ax.set_xlabel('Time [$\mu$s]')
ax.legend(bbox_to_anchor=(0., 1, 1., .102), loc='lower left',ncols=3, mode="expand", borderaxespad=0., handlelength=1.5)

ax = axes['a']
R_opt, pdf_y, pdf_x, _, _ = f.resolving_power(H_opt, binsize)
print(R_opt)
ax.plot(pdf_x, pdf_y, c='k', ls='--', label='KDE', lw=1)
ax.annotate('Noise', xy=(0.2, 350), xytext=(0.5, 350),
            arrowprops=dict(facecolor='black', shrink=0.05))
ax.annotate('Secondary', xy=(.35, 75), xytext=(.5, 150),
            arrowprops=dict(facecolor='black', shrink=0.05))
ax.annotate('Main', xy=(1.4, 250), xytext=(.75, 250),
            arrowprops=dict(facecolor='black', shrink=0.05))
# END: Add annotations to Gaussian plots
ax.legend(bbox_to_anchor=(0., 1, 1., .102), loc='lower left',
        ncols=4, mode="expand", borderaxespad=0.)
fig.suptitle('3.8 $\mu$m', fontsize=10)
# plt.savefig('resolving power 38umv5.pdf')

In [ ]:
def single_gaussian(x, mu1, sigma1, A1):
    return A1 * norm.pdf(x, mu1, sigma1)

binsizes = np.arange(0.005, 0.15, 0.001)
Rs = []
fig, axes = plt.subplot_mosaic('abc', figsize=(10,3))
for i, binsize  in enumerate(binsizes):
    bins=np.arange(-.5, 2, binsize)
    R, pdf_y, pdf_x, _, _ = f.resolving_power(H_opt, binsize)
    Rs.append(R)
    if i==0:
        ax = axes['a']
        ax.hist(H_opt, bins=bins, zorder=0)
        ax.plot(pdf_x, pdf_y, c='k', ls=':', label='KDE', lw=1.5)
    if i==len(binsizes)-1:
        ax = axes['b']
        ax.hist(H_opt, bins=bins, zorder=0)
        ax.plot(pdf_x, pdf_y, c='k', ls=':', label='KDE', lw=1.5)
    print(R)
    ax = axes['c']
    ax.scatter(binsize, R)
print(np.mean(Rs), np.sqrt(1/12*(np.amax(Rs)-np.amin(Rs))**2), np.std(Rs))
axes['c'].plot(np.mean(Rs))
axes['c'].plot(np.mean(Rs))